# 02 — Preprocessing & Feature Engineering

**Goal:** Clean the dataset, engineer features, encode categoricals, and produce train/test splits for both models.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
import joblib
import warnings

warnings.filterwarnings('ignore')

## 1. Load Raw Data

In [ ]:
df = pd.read_csv('../data/dpwh_flood_control_projects.csv')
print(f'Raw shape: {df.shape}')
df.head()

## 2. Parse Data Types

In [ ]:
# Convert dates
df['ActualCompletionDate'] = pd.to_datetime(df['ActualCompletionDate'], errors='coerce')
df['StartDate'] = pd.to_datetime(df['StartDate'], errors='coerce')

# Convert numerics
df['ApprovedBudgetForContract'] = pd.to_numeric(df['ApprovedBudgetForContract'], errors='coerce')
df['ContractCost'] = pd.to_numeric(df['ContractCost'], errors='coerce')

print('Date columns converted.')
print(f'Rows with valid dates: {df.dropna(subset=["StartDate", "ActualCompletionDate"]).shape[0]}')

## 3. Feature Engineering

In [ ]:
# --- Duration ---
df['DurationDays'] = (df['ActualCompletionDate'] - df['StartDate']).dt.days

# --- Cost Variance ---
df['BudgetDifference'] = df['ContractCost'] - df['ApprovedBudgetForContract']

# --- Budget Ratio ---
df['BudgetRatio'] = df['ContractCost'] / df['ApprovedBudgetForContract']

# --- Year (already exists as FundingYear) ---
# df['ProjectYear'] = df['FundingYear']  # Already available

# --- Delay Target (for classification) ---
# A project is "delayed" if DurationDays exceeds the median duration
# OR if we can compute planned vs actual (we only have actual completion date)
# Strategy: Use DurationDays > median as proxy for delay
median_duration = df['DurationDays'].median()
df['IsDelayed'] = (df['DurationDays'] > median_duration).astype(int)

print(f'Median duration: {median_duration} days')
print(f'Delayed projects: {df["IsDelayed"].sum()} / {len(df)} ({df["IsDelayed"].mean()*100:.1f}%)')
print(f'\nNew columns: DurationDays, BudgetDifference, BudgetRatio, IsDelayed')

## 4. Drop Rows with Critical Missing Values

In [ ]:
critical_cols = [
    'Region', 'Province', 'TypeOfWork', 'FundingYear',
    'ApprovedBudgetForContract', 'ContractCost',
    'Contractor', 'StartDate', 'ActualCompletionDate',
    'DurationDays'
]

before = len(df)
df_clean = df.dropna(subset=critical_cols).copy()
after = len(df_clean)

# Remove unreasonable values
df_clean = df_clean[df_clean['DurationDays'] > 0]
df_clean = df_clean[df_clean['ContractCost'] > 0]
df_clean = df_clean[df_clean['ApprovedBudgetForContract'] > 0]

print(f'Rows before cleaning: {before}')
print(f'Rows after cleaning: {len(df_clean)}')
print(f'Dropped: {before - len(df_clean)} rows')

## 5. Encode Categorical Features

In [ ]:
# Define categorical columns to encode
cat_cols = ['Region', 'Province', 'Municipality', 'TypeOfWork', 'Contractor', 'MainIsland']

# Fill missing Municipality with 'Unknown'
df_clean['Municipality'] = df_clean['Municipality'].fillna('Unknown')

# Use OrdinalEncoder with OOV handling
encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

encoded_cols = [f'{col}_encoded' for col in cat_cols]
df_clean[encoded_cols] = encoder.fit_transform(df_clean[cat_cols])

print(f'Encoded {len(cat_cols)} categorical columns.')
print(f'Categories per column:')
for col in cat_cols:
    print(f'  {col}: {df_clean[col].nunique()} unique values')

## 6. Define Feature Sets for Each Model

In [ ]:
# --- Features for DELAY prediction (classification) ---
delay_features = [
    'Region_encoded', 'Province_encoded', 'Municipality_encoded',
    'TypeOfWork_encoded', 'Contractor_encoded', 'MainIsland_encoded',
    'ApprovedBudgetForContract', 'ContractorCount', 'FundingYear'
]

delay_target = 'IsDelayed'

# --- Features for BUDGET prediction (regression) ---
budget_features = [
    'Region_encoded', 'Province_encoded', 'Municipality_encoded',
    'TypeOfWork_encoded', 'MainIsland_encoded',
    'FundingYear'
]

budget_target = 'ContractCost'

print('Delay model features:', delay_features)
print(f'Delay target: {delay_target}')
print()
print('Budget model features:', budget_features)
print(f'Budget target: {budget_target}')

## 7. Train/Test Split

In [ ]:
# --- Delay model split ---
X_delay = df_clean[delay_features]
y_delay = df_clean[delay_target]

X_delay_train, X_delay_test, y_delay_train, y_delay_test = train_test_split(
    X_delay, y_delay, test_size=0.2, random_state=42, stratify=y_delay
)

print(f'Delay model — Train: {X_delay_train.shape}, Test: {X_delay_test.shape}')
print(f'Class distribution (train): {y_delay_train.value_counts().to_dict()}')

# --- Budget model split ---
X_budget = df_clean[budget_features]
y_budget = df_clean[budget_target]

X_budget_train, X_budget_test, y_budget_train, y_budget_test = train_test_split(
    X_budget, y_budget, test_size=0.2, random_state=42
)

print(f'\nBudget model — Train: {X_budget_train.shape}, Test: {X_budget_test.shape}')

## 8. Save Preprocessed Data & Encoder

In [ ]:
import os
os.makedirs('../model_artifacts', exist_ok=True)

# Save encoder
joblib.dump(encoder, '../model_artifacts/encoder.pkl')
print('Saved: encoder.pkl')

# Save the feature lists for reference
feature_config = {
    'delay_features': delay_features,
    'budget_features': budget_features,
    'cat_cols': cat_cols,
    'encoded_cols': encoded_cols,
    'delay_target': delay_target,
    'budget_target': budget_target,
    'median_duration': median_duration,
}
joblib.dump(feature_config, '../model_artifacts/feature_config.pkl')
print('Saved: feature_config.pkl')

# Save processed dataframes for next notebooks
df_clean.to_parquet('../model_artifacts/processed_data.parquet', index=False)
print('Saved: processed_data.parquet')

print(f'\nAll artifacts saved to ../model_artifacts/')